# 03 Model Outputs Summary

Repository notebook for summarizing baseline, SACU-agent, fusion, ablation, threshold, and bootstrap outputs.

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

def find_project_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "README.md").exists() and (candidate / "src").exists():
            return candidate
    return Path.cwd()

PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Project root: {PROJECT_ROOT}")

def read_optional_table(path: Path) -> pd.DataFrame:
    if not path.exists():
        print(f"Missing file: {path}")
        return pd.DataFrame()
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() == ".tsv":
        return pd.read_csv(path, sep="\t")
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    if path.suffix.lower() in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    raise ValueError(f"Unsupported table format: {path.suffix}")

def find_files(patterns, roots):
    rows = []
    for root in roots:
        root = Path(root)
        if not root.exists():
            continue
        for pattern in patterns:
            for path in root.rglob(pattern):
                rows.append({
                    "file": path.name,
                    "path": str(path.relative_to(PROJECT_ROOT)) if PROJECT_ROOT in path.parents else str(path),
                    "size_kb": round(path.stat().st_size / 1024, 2),
                    "modified": pd.to_datetime(path.stat().st_mtime, unit="s"),
                })
    return pd.DataFrame(rows).sort_values("modified", ascending=False).reset_index(drop=True) if rows else pd.DataFrame()

## Locate model and evaluation outputs

In [ ]:
output_files = find_files(
    patterns=["*.csv", "*.json", "*.txt"],
    roots=[PROJECT_ROOT / "outputs", PROJECT_ROOT / "results"],
)

model_related = output_files[
    output_files["file"].str.contains(
        "baseline|sacu|agent|fusion|ablation|threshold|bootstrap|coordination|interpretability|metric|prediction",
        case=False,
        regex=True,
        na=False,
    )
].reset_index(drop=True) if not output_files.empty else pd.DataFrame()

model_related

## Load key result tables

In [ ]:
def load_first_matching(name_pattern: str) -> pd.DataFrame:
    if model_related.empty:
        return pd.DataFrame()
    matches = model_related[
        model_related["file"].str.contains(name_pattern, case=False, regex=True, na=False)
        & model_related["file"].str.endswith(".csv")
    ]
    if matches.empty:
        print(f"No CSV found for pattern: {name_pattern}")
        return pd.DataFrame()
    path = PROJECT_ROOT / matches.iloc[0]["path"]
    print(f"Loading {path}")
    return read_optional_table(path)

baseline_df = load_first_matching("baseline.*performance|baseline.*metric|Stage2C")
sacu_df = load_first_matching("sacu.*test|fusion.*summary|Stage2D")
threshold_df = load_first_matching("threshold|operating")
ablation_df = load_first_matching("ablation")
bootstrap_df = load_first_matching("bootstrap|ci")
interpretability_df = load_first_matching("interpretability|pathway.*contribution")

{
    "baseline": baseline_df.shape,
    "sacu": sacu_df.shape,
    "threshold": threshold_df.shape,
    "ablation": ablation_df.shape,
    "bootstrap": bootstrap_df.shape,
    "interpretability": interpretability_df.shape,
}

## Performance table builder

In [ ]:
metric_cols = [
    "model", "agent_name", "coordination_mechanism", "operating_point",
    "roc_auc", "average_precision", "accuracy", "balanced_accuracy",
    "sensitivity", "specificity", "precision", "recall", "f1", "mcc",
]

frames = []
for name, df in {
    "baseline": baseline_df,
    "sacu": sacu_df,
    "threshold": threshold_df,
    "ablation": ablation_df,
}.items():
    if not df.empty:
        tmp = df.copy()
        tmp["source_table"] = name
        available = ["source_table"] + [c for c in metric_cols if c in tmp.columns]
        frames.append(tmp[available])

performance_summary = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
performance_summary

## Best reported model by ROC-AUC

In [ ]:
if not performance_summary.empty and "roc_auc" in performance_summary.columns:
    best_by_auc = performance_summary.dropna(subset=["roc_auc"]).sort_values("roc_auc", ascending=False).head(20)
else:
    best_by_auc = pd.DataFrame()

best_by_auc

## Bootstrap confidence interval summary

In [ ]:
if not bootstrap_df.empty:
    ci_cols = [
        "model", "metric", "point_estimate", "ci_lower", "ci_upper",
        "confidence_level", "n_bootstrap_valid"
    ]
    bootstrap_summary = bootstrap_df[[c for c in ci_cols if c in bootstrap_df.columns]].copy()
else:
    bootstrap_summary = pd.DataFrame()

bootstrap_summary.head(50)

## Interpretability and pathway contribution summary

In [ ]:
interpretability_df